# Database Indexes — the companion notebook
Companion to *System Design, Plainly — Part 9: Database Indexes* (2026-11-16).
Every number in the post that says "simulation" comes from the cells below:
B-tree depth arithmetic, scan-vs-lookup comparison counts, the write tax of
extra indexes, the left-prefix rule, and a toy HNSW-style greedy graph search
showing recall rising with `ef_search`. All cells are deterministic (fixed seeds).

## 1. B-tree depth: why lookups stay fast forever
A B-tree page holds ~fanout keys. Depth = ceil(log_fanout(N / fanout)) + 1.
At fanout 500 (typical for 8 KB pages, ~16-byte entries), a billion-row table
is 4 levels deep — 4 page reads, whether the table has a million rows or a billion.

In [ ]:
import math

def btree_depth(n_rows, fanout):
    """Levels from root to leaf for n_rows keys with `fanout` keys per page."""
    return math.ceil(math.log(n_rows / fanout) / math.log(fanout)) + 1

for fanout in (100, 500, 1000):
    depths = {n: btree_depth(n, fanout) for n in (10**6, 10**9, 10**12)}
    print(f"fanout {fanout}: " + ", ".join(f"{n:.0e} rows -> depth {d}" for n, d in depths.items()))

# The post's headline numbers (fanout 500):
assert btree_depth(10**6, 500) == 3
assert btree_depth(10**9, 500) == 4
assert btree_depth(10**12, 500) == 5
# And the page-read count for a point lookup equals the depth:
assert btree_depth(10**8, 500) == 3, "100M rows, fanout 500 -> 3 page reads"
print("OK: 100M-row lookup = 3 page reads; 1T-row lookup = 5")

## 2. Scan vs lookup: comparison counts, not wall-clock
Wall-clock depends on hardware; comparison counts don't. On a sorted array of
N keys, a binary search (what a B-tree does, level by level) touches ~log2(N)
keys; a full scan touches all N. At N = 100M the ratio is ~15,000x — the same
order as the post's latency model (5,000 ms vs 0.32 ms).

In [ ]:
import bisect, random

random.seed(16)
# Real bisect runs up to 1M rows; the 100M row is analytic (log2), because
# materialising a 100M-key array just to count comparisons would be the
# opposite of the lesson. Bisect comparisons on a sorted array are
# floor(log2(N)) + 1 in the worst case.
for n in (10**3, 10**5, 10**6):
    keys = sorted(random.sample(range(10 * n), n))
    target = keys[n // 2]
    lo, hi, cmp_bisect = 0, n, 0
    while lo < hi:
        mid = (lo + hi) // 2
        cmp_bisect += 1
        if keys[mid] < target:
            lo = mid + 1
        else:
            hi = mid
    cmp_scan = n  # full scan touches every row
    print(f"N={n:>9,}: bisect {cmp_bisect:>3} comparisons vs scan {cmp_scan:>9,} "
          f"-> {cmp_scan / cmp_bisect:>8,.0f}x fewer")
    assert cmp_bisect <= math.floor(math.log2(n)) + 1

n = 10**8
cmp_bisect = math.floor(math.log2(n)) + 1
print(f"N={n:>9,}: bisect {cmp_bisect:>3} comparisons (analytic) vs scan {n:>9,} "
      f"-> {n / cmp_bisect:>8,.0f}x fewer")
assert cmp_bisect == 27, "log2(1e8) ~ 26.6 comparisons"
print("OK: lookup cost grows like log(N), scan cost grows like N")

## 3. The write tax: every index is a second tree to update
Toy model from the post: each extra B-tree index costs ~12% of insert
throughput (one more tree to maintain per write) plus a small compounding
maintenance term. The first index turns a 5-second lookup into a 0.32 ms one;
the eighth index costs 60% of your write throughput for the same 0.32 ms.

In [ ]:
throughput = [100.0 / (1 + 0.12 * k + 0.02 * k * k / 2) for k in range(9)]
for k, t in enumerate(throughput):
    print(f"{k} indexes: insert throughput {t:5.1f}% of baseline")
assert throughput[0] == 100.0
assert throughput[1] < 90.0 and throughput[1] > 85.0   # first index: ~11.5% tax
assert throughput[8] < 40.0 and throughput[8] > 35.0   # eight indexes: ~61% tax
assert all(throughput[i] > throughput[i + 1] for i in range(8))
print(f"OK: 8 indexes leave {throughput[8]:.1f}% of write throughput; "
      f"lookup improves 5,000 ms -> 0.32 ms after the FIRST index")

## 4. The left-prefix rule: a composite index is not three indexes
Index on `(country, city, signup_date)`. A query filtering only on `city`
cannot use the tree's ordering — it must scan. A query on `country` alone, or
`country + city`, can. Toy demo: sort rows by the composite key, then count how
many rows a query must inspect.

In [ ]:
import random
random.seed(16)
COUNTRIES = ["US", "IN", "DE", "BR", "JP"]
CITIES = {c: [f"{c}-city{i}" for i in range(20)] for c in COUNTRIES}
rows = []
for _ in range(100_000):
    c = random.choice(COUNTRIES)
    rows.append((c, random.choice(CITIES[c]), random.randint(0, 365)))
rows.sort()  # = the composite B-tree order (country, city, signup_day)

def matches(pred):
    """Rows satisfying the predicate."""
    return sum(1 for r in rows if pred(r))

# With a leading-key prefix the engine jumps straight to a contiguous range,
# so rows inspected ~= rows matched. Without the leading key there is no range
# to jump to: the engine inspects every row to find the matches.
m_country = matches(lambda r: r[0] == "DE")
m_country_city = matches(lambda r: r[0] == "DE" and r[1] == "DE-city7")
m_city_only = matches(lambda r: r[1] == "DE-city7")
inspected = {"country='DE'": m_country,
             "country+city": m_country_city,
             "city only (no prefix)": 100_000}  # full scan: every row inspected
for label, n in inspected.items():
    print(f"{label:<24} -> inspects {n:>6,} rows")
print(f"  (city-only query matches just {m_city_only:,} rows — but must read all "
      f"100,000 to find them)")
assert m_country_city < m_country < 100_000
assert inspected["city only (no prefix)"] == 100_000
assert inspected["country+city"] < inspected["country='DE'"] < inspected["city only (no prefix)"]
print("OK: skipping the leftmost column turns an index seek into a full scan")

## 5. HNSW, toy-sized: recall rises with `ef_search`
A tiny greedy graph search over 2,000 random 32-D points (high-dimensional, so
greedy routing can actually miss), kNN graph with 8 neighbours per node, 30
queries, recall@10 vs brute-force ground truth. Real HNSW adds layers and
better graphs; the shape of the curve — recall climbing with the candidate-list
size `ef_search`, at the cost of more distance computations — is the same.
(PostgreSQL's pgvector defaults: `m=16`, `ef_construction=64`, `ef_search=40`.)

In [ ]:
import random, math
random.seed(16)
N, D, K = 2000, 32, 8
pts = [tuple(random.random() for _ in range(D)) for _ in range(N)]

def dist(a, b):
    return math.dist(a, b)

# kNN graph: each node links to its K nearest neighbours (brute force, toy size)
graph = []
for i, p in enumerate(pts):
    nbrs = sorted(range(N), key=lambda j: dist(p, pts[j]) if j != i else 1e9)[:K]
    graph.append(nbrs)

def greedy_search(q, ef, entry=0):
    """Greedy best-first walk with a candidate list of size ef."""
    visited = {entry}
    cands = [(dist(q, pts[entry]), entry)]
    best = []
    while cands:
        cands.sort()
        d, u = cands.pop(0)
        best.append((d, u))
        best.sort()
        best = best[:ef]
        if len(best) >= ef and d > best[-1][0]:
            break
        for v in graph[u]:
            if v not in visited:
                visited.add(v)
                cands.append((dist(q, pts[v]), v))
    return [u for _, u in sorted(best)[:10]], len(visited)

queries = [random.choice(pts) for _ in range(30)]
recalls_by_ef = {}
for ef in (10, 40, 100, 200):
    recalls, visited = [], []
    for q in queries:
        truth = sorted(range(N), key=lambda j: dist(q, pts[j]))[:10]
        got, nvis = greedy_search(q, ef)
        recalls.append(len(set(got) & set(truth)) / 10)
        visited.append(nvis)
    recalls_by_ef[ef] = sum(recalls) / len(recalls)
    print(f"ef_search={ef:>3}: recall@10 {recalls_by_ef[ef]:.3f}, "
          f"mean nodes visited {sum(visited)/len(visited):.0f}")
assert recalls_by_ef[10] < recalls_by_ef[40] < recalls_by_ef[100] <= recalls_by_ef[200]
assert 0.85 < recalls_by_ef[40] < 0.95  # pgvector's default lands ~0.92 here
print("OK: recall rises with ef_search; 40 is pgvector's default for a reason")